In [ ]:
# imports

from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# paths and constants

PROCESSED_DIR = Path("processed_data")
MODEL_RESULTS_DIR = Path("results") / "final_models"
WAKE_RESULTS_DIR = Path("results") / "wake_analysis"

WAKE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"
STATIC_FILE = PROCESSED_DIR / "penmanshiel_static.parquet"

SEEDS = [1, 21, 42, 84, 123]

MIN_DOWNSTREAM_D = 2.0
JENSEN_K = 0.075
JENSEN_CT = 0.8

DIRECTION_BIN_SIZE = 5
MIN_OBSERVATIONS_PER_DIRECTION = 20

WIND_SPEED_BINS = [4, 6, 8, 10, 12, 15, np.inf]
WIND_SPEED_LABELS = [
    "4–6",
    "6–8",
    "8–10",
    "10–12",
    "12–15",
    "15+",
]

TABLE_WIND_SPEED_LABELS = WIND_SPEED_LABELS[:-1]


In [ ]:
# load cleaned data and final model predictions

scada = pd.read_parquet(SCADA_FILE).copy()
static = pd.read_parquet(STATIC_FILE).copy()

scada["timestamp"] = pd.to_datetime(
    scada["timestamp"],
    utc=True,
)

static["turbine_id"] = static["turbine_id"].astype(int)
turbine_ids = sorted(static["turbine_id"].unique())

prediction_data = {}

for seed in SEEDS:
    file_path = (
        MODEL_RESULTS_DIR
        / f"predictions_seed_{seed}.parquet"
    )

    prediction_data[seed] = pd.read_parquet(
        file_path
    ).copy()

    prediction_data[seed].index = pd.to_datetime(
        prediction_data[seed].index,
        utc=True,
    )

reference_seed = SEEDS[0]
reference_predictions = prediction_data[reference_seed]

for seed in SEEDS[1:]:
    if not reference_predictions.index.equals(
        prediction_data[seed].index
    ):
        raise ValueError(
            f"test timestamps do not align for seed {seed}"
        )

    if not np.allclose(
        reference_predictions["actual_farm_power"],
        prediction_data[seed]["actual_farm_power"],
    ):
        raise ValueError(
            f"observed farm power differs for seed {seed}"
        )

print(f"test timestamps: {len(reference_predictions):,}")
print(f"turbines: {turbine_ids}")


In [ ]:
# wind direction and common test-period table

def direction_from_components(
    wind_direction_sine,
    wind_direction_cosine,
):
    return (
        np.degrees(
            np.arctan2(
                wind_direction_sine,
                wind_direction_cosine,
            )
        )
        + 360
    ) % 360


common_test = pd.DataFrame(
    index=reference_predictions.index
)

common_test["global_ws"] = (
    reference_predictions["global_ws"]
)

common_test["wind_direction"] = direction_from_components(
    reference_predictions["wd_sin"].to_numpy(),
    reference_predictions["wd_cos"].to_numpy(),
)

common_test["actual_farm_power"] = (
    reference_predictions["actual_farm_power"]
)

common_test["wind_dir_1deg"] = (
    np.round(common_test["wind_direction"]) % 360
).astype(int)

common_test["dir_bin"] = (
    np.round(
        common_test["wind_direction"]
        / DIRECTION_BIN_SIZE
    )
    * DIRECTION_BIN_SIZE
) % 360


In [ ]:
# build direction-dependent pair geometry from the turbine layout

def build_geometry_lookup(
    static,
    direction_bins=np.arange(0, 360, 1),
):
    layout = (
        static[
            [
                "turbine_id",
                "x_m",
                "y_m",
                "rotor_diameter_m",
            ]
        ]
        .copy()
        .set_index("turbine_id")
    )

    rows = []

    for wind_direction in direction_bins:
        theta = np.deg2rad(wind_direction)

        # wind direction is the direction from which the wind arrives
        downstream_unit = np.array(
            [-np.sin(theta), -np.cos(theta)]
        )

        lateral_unit = np.array(
            [np.cos(theta), -np.sin(theta)]
        )

        for turbine_i in layout.index:
            origin = layout.loc[
                turbine_i,
                ["x_m", "y_m"],
            ].to_numpy(dtype=float)

            rotor_diameter = float(
                layout.loc[
                    turbine_i,
                    "rotor_diameter_m",
                ]
            )

            for turbine_j in layout.index:
                if turbine_i == turbine_j:
                    continue

                target = layout.loc[
                    turbine_j,
                    ["x_m", "y_m"],
                ].to_numpy(dtype=float)

                displacement = target - origin

                downstream_distance = (
                    displacement @ downstream_unit
                )

                lateral_distance = (
                    displacement @ lateral_unit
                )

                rows.append(
                    {
                        "wind_dir_bin": int(wind_direction),
                        "turbine_i": int(turbine_i),
                        "turbine_j": int(turbine_j),
                        "downstream_D": (
                            downstream_distance
                            / rotor_diameter
                        ),
                        "lateral_D": (
                            lateral_distance
                            / rotor_diameter
                        ),
                    }
                )

    return pd.DataFrame(rows)


geometry_lookup = build_geometry_lookup(static)

geometry_lookup.to_parquet(
    WAKE_RESULTS_DIR / "geometry_lookup.parquet",
    index=False,
)

print(
    f"geometry rows: {len(geometry_lookup):,}"
)


In [ ]:
# identify Jensen-native wake pairs

def jensen_deficit(
    downstream_D,
    lateral_D,
    k=JENSEN_K,
    Ct=JENSEN_CT,
):
    downstream_D = np.asarray(
        downstream_D,
        dtype=float,
    )

    lateral_D = np.abs(
        np.asarray(
            lateral_D,
            dtype=float,
        )
    )

    wake_radius_D = (
        0.5
        + k * downstream_D
    )

    centre_deficit = (
        1 - np.sqrt(1 - Ct)
    ) / (
        1 + 2 * k * downstream_D
    ) ** 2

    inside_wake = (
        np.isfinite(downstream_D)
        & np.isfinite(lateral_D)
        & (downstream_D >= MIN_DOWNSTREAM_D)
        & (lateral_D <= wake_radius_D)
    )

    return np.where(
        inside_wake,
        centre_deficit,
        0.0,
    )


geometry_lookup["jensen_radius_D"] = (
    0.5
    + JENSEN_K
    * geometry_lookup["downstream_D"]
)

geometry_lookup["jensen_deficit"] = jensen_deficit(
    geometry_lookup["downstream_D"],
    geometry_lookup["lateral_D"],
)

geometry_lookup["is_wake_pair"] = (
    geometry_lookup["jensen_deficit"] > 0
)

wake_pairs = geometry_lookup.loc[
    geometry_lookup["is_wake_pair"]
].copy()

wake_pairs.to_parquet(
    WAKE_RESULTS_DIR / "jensen_wake_pairs.parquet",
    index=False,
)

print(f"wake pair-direction rows: {len(wake_pairs):,}")
print(
    "unique direction × downstream-turbine cases: "
    f"{wake_pairs[['wind_dir_bin', 'turbine_j']].drop_duplicates().shape[0]:,}"
)
print(
    f"identified downstream range: "
    f"{wake_pairs['downstream_D'].min():.3f}–"
    f"{wake_pairs['downstream_D'].max():.3f} D"
)


In [ ]:
# nearest upstream Jensen interaction for each direction and turbine

nearest_wake_pairs = (
    wake_pairs
    .sort_values("downstream_D")
    .drop_duplicates(
        ["wind_dir_bin", "turbine_j"]
    )
    .copy()
)

direction_turbine_grid = (
    pd.MultiIndex.from_product(
        [
            np.arange(0, 360),
            turbine_ids,
        ],
        names=[
            "wind_dir_1deg",
            "turbine_id",
        ],
    )
    .to_frame(index=False)
)

nearest_lookup = (
    nearest_wake_pairs[
        [
            "wind_dir_bin",
            "turbine_j",
            "jensen_deficit",
        ]
    ]
    .rename(
        columns={
            "wind_dir_bin": "wind_dir_1deg",
            "turbine_j": "turbine_id",
        }
    )
)

direction_turbine_grid = (
    direction_turbine_grid
    .merge(
        nearest_lookup,
        on=["wind_dir_1deg", "turbine_id"],
        how="left",
    )
)

direction_turbine_grid["jensen_deficit"] = (
    direction_turbine_grid["jensen_deficit"]
    .fillna(0.0)
)

direction_turbine_grid["is_waked"] = (
    direction_turbine_grid["jensen_deficit"] > 0
)


In [ ]:
# timestamp × turbine wake table

test_grid = (
    common_test[
        ["wind_dir_1deg"]
    ]
    .reset_index(names="timestamp")
    .merge(
        direction_turbine_grid,
        on="wind_dir_1deg",
        how="left",
    )
)

test_scada = scada[
    scada["timestamp"].isin(
        common_test.index
    )
][
    [
        "timestamp",
        "turbine_id",
        "wind_speed",
    ]
].copy()

test_grid = test_grid.merge(
    test_scada,
    on=["timestamp", "turbine_id"],
    how="left",
    validate="one_to_one",
)

positive_jensen_median = (
    test_grid.loc[
        test_grid["jensen_deficit"] > 0,
        "jensen_deficit",
    ]
    .median()
)

test_grid["wake_class"] = "Low"

test_grid.loc[
    (
        (test_grid["jensen_deficit"] > 0)
        & (
            test_grid["jensen_deficit"]
            <= positive_jensen_median
        )
    ),
    "wake_class",
] = "Medium"

test_grid.loc[
    test_grid["jensen_deficit"]
    > positive_jensen_median,
    "wake_class",
] = "High"

test_grid["wake_class"] = pd.Categorical(
    test_grid["wake_class"],
    categories=["Low", "Medium", "High"],
    ordered=True,
)

test_grid.to_parquet(
    WAKE_RESULTS_DIR
    / "test_turbine_wake_classes.parquet",
    index=False,
)

print(
    f"median positive Jensen deficit: "
    f"{positive_jensen_median:.6f}"
)

print(
    test_grid["wake_class"]
    .value_counts()
    .sort_index()
    .to_string()
)


In [ ]:
# turbine-level Model B and Model C error by wake class and wind-speed band

wake_lookup = test_grid[
    [
        "timestamp",
        "turbine_id",
        "jensen_deficit",
        "wake_class",
    ]
].copy()

B_seed_rows = []
C_seed_rows = []

for seed in SEEDS:
    predictions = prediction_data[seed]

    long_parts = []

    for turbine_id in turbine_ids:
        long_parts.append(
            pd.DataFrame(
                {
                    "timestamp": predictions.index,
                    "turbine_id": turbine_id,
                    "global_ws": predictions["global_ws"].to_numpy(),
                    "B_abs_error": np.abs(
                        predictions[
                            f"pred_B_ws_t{turbine_id}"
                        ].to_numpy()
                        - predictions[
                            f"actual_ws_t{turbine_id}"
                        ].to_numpy()
                    ),
                    "C_abs_error": np.abs(
                        predictions[
                            f"pred_C_power_t{turbine_id}"
                        ].to_numpy()
                        - predictions[
                            f"actual_power_t{turbine_id}"
                        ].to_numpy()
                    ),
                }
            )
        )

    long_errors = pd.concat(
        long_parts,
        ignore_index=True,
    )

    long_errors = long_errors.merge(
        wake_lookup,
        on=["timestamp", "turbine_id"],
        how="left",
        validate="one_to_one",
    )

    long_errors["ws_bin"] = pd.cut(
        long_errors["global_ws"],
        bins=WIND_SPEED_BINS,
        labels=WIND_SPEED_LABELS,
        right=False,
    )

    B_seed = (
        long_errors
        .dropna(subset=["ws_bin", "wake_class"])
        .groupby(
            ["ws_bin", "wake_class"],
            observed=True,
        )
        .agg(
            MAE=("B_abs_error", "mean"),
            n=("B_abs_error", "size"),
        )
        .reset_index()
    )

    B_seed["seed"] = seed
    B_seed_rows.append(B_seed)

    C_seed = (
        long_errors
        .dropna(subset=["ws_bin", "wake_class"])
        .groupby(
            ["ws_bin", "wake_class"],
            observed=True,
        )
        .agg(
            MAE=("C_abs_error", "mean"),
            n=("C_abs_error", "size"),
        )
        .reset_index()
    )

    C_seed["seed"] = seed
    C_seed_rows.append(C_seed)

B_seed_results = pd.concat(
    B_seed_rows,
    ignore_index=True,
)

C_seed_results = pd.concat(
    C_seed_rows,
    ignore_index=True,
)

B_wake_summary = (
    B_seed_results
    .groupby(
        ["ws_bin", "wake_class"],
        observed=True,
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_sd=("MAE", "std"),
    )
    .reset_index()
)

C_wake_summary = (
    C_seed_results
    .groupby(
        ["ws_bin", "wake_class"],
        observed=True,
    )
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_sd=("MAE", "std"),
    )
    .reset_index()
)

B_seed_results.to_csv(
    WAKE_RESULTS_DIR
    / "model_B_wake_error_by_seed.csv",
    index=False,
)

C_seed_results.to_csv(
    WAKE_RESULTS_DIR
    / "model_C_wake_error_by_seed.csv",
    index=False,
)

B_wake_summary.to_csv(
    WAKE_RESULTS_DIR
    / "model_B_wake_error_summary.csv",
    index=False,
)

C_wake_summary.to_csv(
    WAKE_RESULTS_DIR
    / "model_C_wake_error_summary.csv",
    index=False,
)

print(B_wake_summary.round(4).to_string(index=False))
print()
print(C_wake_summary.round(3).to_string(index=False))


In [ ]:
# training-only empirical turbine power curves for the no-wake reference

training_end = pd.Timestamp(
    "2021-06-10",
    tz="UTC",
)

training_scada = scada[
    scada["timestamp"] < training_end
].copy()

power_curve_bins = np.arange(
    0,
    30.5,
    0.5,
)

power_curves = {}

for turbine_id in turbine_ids:
    turbine_data = training_scada[
        training_scada["turbine_id"] == turbine_id
    ].copy()

    turbine_data["ws_bin"] = pd.cut(
        turbine_data["wind_speed"],
        bins=power_curve_bins,
        include_lowest=True,
    )

    median_power = (
        turbine_data
        .groupby(
            "ws_bin",
            observed=True,
        )["power_kw"]
        .median()
        .dropna()
    )

    power_curves[turbine_id] = (
        np.array(
            [
                interval.mid
                for interval in median_power.index
            ],
            dtype=float,
        ),
        median_power.to_numpy(dtype=float),
    )


In [ ]:
# Jensen-defined non-waked reference wind speed

free_reference_ws = (
    test_grid.loc[
        ~test_grid["is_waked"]
    ]
    .groupby("timestamp")["wind_speed"]
    .median()
    .rename("reference_ws")
)

relative_base = (
    common_test
    .reset_index(names="timestamp")
    .merge(
        free_reference_ws,
        on="timestamp",
        how="left",
    )
    .dropna(subset=["reference_ws"])
    .copy()
)

reference_power = np.zeros(
    (
        len(relative_base),
        len(turbine_ids),
    )
)

for turbine_index, turbine_id in enumerate(
    turbine_ids
):
    wind_speed_midpoints, power_values = (
        power_curves[turbine_id]
    )

    reference_power[:, turbine_index] = np.interp(
        relative_base["reference_ws"].to_numpy(),
        wind_speed_midpoints,
        power_values,
        left=0.0,
        right=power_values[-1],
    )

relative_base["no_wake_power"] = (
    reference_power.sum(axis=1)
)

relative_base = relative_base[
    relative_base["no_wake_power"] > 0
].copy()

missing_reference_timestamps = (
    len(common_test)
    - relative_base["timestamp"].nunique()
)

print(
    f"missing no-wake reference timestamps: "
    f"{missing_reference_timestamps}"
)


In [ ]:
# directional relative farm power and agreement metrics

relative_seed_rows = []
agreement_rows = []

for seed in SEEDS:
    predictions = prediction_data[seed]

    seed_predictions = pd.DataFrame(
        {
            "timestamp": predictions.index,
            "Model A": predictions[
                "pred_A_power"
            ].to_numpy(),
            "Model B": predictions[
                "pred_B_power"
            ].to_numpy(),
            "Model C": predictions[
                "pred_C_power"
            ].to_numpy(),
        }
    )

    relative_seed = relative_base.merge(
        seed_predictions,
        on="timestamp",
        how="left",
        validate="one_to_one",
    )

    grouped = (
        relative_seed
        .groupby("dir_bin")
        .agg(
            n_obs=("timestamp", "size"),
            actual_sum=("actual_farm_power", "sum"),
            A_sum=("Model A", "sum"),
            B_sum=("Model B", "sum"),
            C_sum=("Model C", "sum"),
            reference_sum=("no_wake_power", "sum"),
        )
        .reset_index()
    )

    grouped = grouped[
        grouped["n_obs"]
        >= MIN_OBSERVATIONS_PER_DIRECTION
    ].copy()

    grouped["Observed"] = (
        grouped["actual_sum"]
        / grouped["reference_sum"]
    )

    grouped["Model A"] = (
        grouped["A_sum"]
        / grouped["reference_sum"]
    )

    grouped["Model B"] = (
        grouped["B_sum"]
        / grouped["reference_sum"]
    )

    grouped["Model C"] = (
        grouped["C_sum"]
        / grouped["reference_sum"]
    )

    grouped["seed"] = seed
    relative_seed_rows.append(grouped)

    for model_name in [
        "Model A",
        "Model B",
        "Model C",
    ]:
        agreement_rows.append(
            {
                "seed": seed,
                "model": model_name,
                "correlation": grouped[
                    "Observed"
                ].corr(
                    grouped[model_name]
                ),
                "directional_MAE": np.mean(
                    np.abs(
                        grouped[model_name]
                        - grouped["Observed"]
                    )
                ),
            }
        )

relative_by_seed = pd.concat(
    relative_seed_rows,
    ignore_index=True,
)

relative_summary = (
    relative_by_seed
    .groupby("dir_bin")
    .agg(
        Observed=("Observed", "first"),
        A_mean=("Model A", "mean"),
        A_sd=("Model A", "std"),
        B_mean=("Model B", "mean"),
        B_sd=("Model B", "std"),
        C_mean=("Model C", "mean"),
        C_sd=("Model C", "std"),
    )
    .reset_index()
)

agreement_by_seed = pd.DataFrame(
    agreement_rows
)

agreement_summary = (
    agreement_by_seed
    .groupby("model")
    .agg(
        correlation_mean=("correlation", "mean"),
        correlation_sd=("correlation", "std"),
        MAE_mean=("directional_MAE", "mean"),
        MAE_sd=("directional_MAE", "std"),
    )
    .reset_index()
)

relative_by_seed.to_csv(
    WAKE_RESULTS_DIR
    / "directional_relative_power_by_seed.csv",
    index=False,
)

relative_summary.to_csv(
    WAKE_RESULTS_DIR
    / "directional_relative_power_summary.csv",
    index=False,
)

agreement_summary.to_csv(
    WAKE_RESULTS_DIR
    / "directional_agreement_summary.csv",
    index=False,
)

print(
    agreement_summary
    .round(4)
    .to_string(index=False)
)


In [ ]:
# mean farm-power residual by wind direction

residual_seed_rows = []

for seed in SEEDS:
    predictions = prediction_data[seed]

    residual_data = pd.DataFrame(
        {
            "dir_bin": common_test[
                "dir_bin"
            ].to_numpy(),
            "A_residual": (
                predictions["pred_A_power"].to_numpy()
                - predictions[
                    "actual_farm_power"
                ].to_numpy()
            ),
            "B_residual": (
                predictions["pred_B_power"].to_numpy()
                - predictions[
                    "actual_farm_power"
                ].to_numpy()
            ),
            "C_residual": (
                predictions["pred_C_power"].to_numpy()
                - predictions[
                    "actual_farm_power"
                ].to_numpy()
            ),
        }
    )

    grouped = (
        residual_data
        .groupby("dir_bin")
        .agg(
            A_residual=("A_residual", "mean"),
            B_residual=("B_residual", "mean"),
            C_residual=("C_residual", "mean"),
        )
        .reset_index()
    )

    grouped["seed"] = seed
    residual_seed_rows.append(grouped)

residual_by_seed = pd.concat(
    residual_seed_rows,
    ignore_index=True,
)

residual_summary = (
    residual_by_seed
    .groupby("dir_bin")
    .agg(
        A_mean=("A_residual", "mean"),
        A_sd=("A_residual", "std"),
        B_mean=("B_residual", "mean"),
        B_sd=("B_residual", "std"),
        C_mean=("C_residual", "mean"),
        C_sd=("C_residual", "std"),
    )
    .reset_index()
)

residual_by_seed.to_csv(
    WAKE_RESULTS_DIR
    / "directional_residuals_by_seed.csv",
    index=False,
)

residual_summary.to_csv(
    WAKE_RESULTS_DIR
    / "directional_residuals_summary.csv",
    index=False,
)


In [ ]:
# farm-level directional Jensen severity

directional_severity = (
    wake_pairs
    .assign(
        dir_bin=(
            np.round(
                wake_pairs["wind_dir_bin"]
                / DIRECTION_BIN_SIZE
            )
            * DIRECTION_BIN_SIZE
        ) % 360
    )
    .groupby("dir_bin")["jensen_deficit"]
    .sum()
    .rename("directional_severity")
    .reset_index()
)

all_direction_bins = pd.DataFrame(
    {
        "dir_bin": np.arange(
            0,
            360,
            DIRECTION_BIN_SIZE,
        )
    }
)

directional_severity = (
    all_direction_bins
    .merge(
        directional_severity,
        on="dir_bin",
        how="left",
    )
)

directional_severity["directional_severity"] = (
    directional_severity[
        "directional_severity"
    ].fillna(0.0)
)

q1 = directional_severity[
    "directional_severity"
].quantile(1 / 3)

q2 = directional_severity[
    "directional_severity"
].quantile(2 / 3)

directional_severity["wake_class"] = np.select(
    [
        directional_severity[
            "directional_severity"
        ] <= q1,
        directional_severity[
            "directional_severity"
        ] <= q2,
    ],
    [
        "Low",
        "Medium",
    ],
    default="High",
)

directional_severity["wake_group"] = np.where(
    directional_severity["wake_class"]
    == "High",
    "High wake",
    "Non-high wake",
)

directional_severity.to_csv(
    WAKE_RESULTS_DIR
    / "directional_jensen_severity.csv",
    index=False,
)

high_wake_bins = directional_severity.loc[
    directional_severity["wake_class"]
    == "High",
    "dir_bin",
].tolist()

print(f"directional tertile q1: {q1:.6f}")
print(f"directional tertile q2: {q2:.6f}")
print(f"high-wake direction bins: {high_wake_bins}")


In [ ]:
# high-wake versus non-high-wake error comparison

common_wake = (
    common_test[
        [
            "global_ws",
            "dir_bin",
        ]
    ]
    .reset_index(names="timestamp")
)

common_wake = common_wake.merge(
    directional_severity[
        [
            "dir_bin",
            "wake_group",
        ]
    ],
    on="dir_bin",
    how="left",
    validate="many_to_one",
)

common_wake["ws_bin"] = pd.cut(
    common_wake["global_ws"],
    bins=[4, 6, 8, 10, 12, 15],
    labels=TABLE_WIND_SPEED_LABELS,
    right=False,
)

farm_error_rows = []
B_ws_error_rows = []

for seed in SEEDS:
    predictions = prediction_data[seed]

    farm_errors = pd.DataFrame(
        {
            "timestamp": predictions.index,
            "actual": predictions[
                "actual_farm_power"
            ].to_numpy(),
            "A": predictions[
                "pred_A_power"
            ].to_numpy(),
            "B": predictions[
                "pred_B_power"
            ].to_numpy(),
            "C": predictions[
                "pred_C_power"
            ].to_numpy(),
        }
    )

    farm_errors = farm_errors.merge(
        common_wake[
            [
                "timestamp",
                "ws_bin",
                "wake_group",
            ]
        ],
        on="timestamp",
        how="left",
        validate="one_to_one",
    )

    for model_name in ["A", "B", "C"]:
        model_rows = farm_errors.dropna(
            subset=["ws_bin", "wake_group"]
        ).copy()

        model_rows["abs_error"] = np.abs(
            model_rows[model_name]
            - model_rows["actual"]
        )

        grouped = (
            model_rows
            .groupby(
                ["ws_bin", "wake_group"],
                observed=True,
            )
            .agg(
                MAE=("abs_error", "mean"),
                n=("abs_error", "size"),
            )
            .reset_index()
        )

        grouped["seed"] = seed
        grouped["model"] = model_name
        farm_error_rows.append(grouped)

    turbine_ws_errors = []

    for turbine_id in turbine_ids:
        turbine_ws_errors.append(
            pd.DataFrame(
                {
                    "timestamp": predictions.index,
                    "abs_error": np.abs(
                        predictions[
                            f"pred_B_ws_t{turbine_id}"
                        ].to_numpy()
                        - predictions[
                            f"actual_ws_t{turbine_id}"
                        ].to_numpy()
                    ),
                }
            )
        )

    turbine_ws_errors = pd.concat(
        turbine_ws_errors,
        ignore_index=True,
    )

    turbine_ws_errors = turbine_ws_errors.merge(
        common_wake[
            [
                "timestamp",
                "ws_bin",
                "wake_group",
            ]
        ],
        on="timestamp",
        how="left",
    )

    grouped_B_ws = (
        turbine_ws_errors
        .dropna(subset=["ws_bin", "wake_group"])
        .groupby(
            ["ws_bin", "wake_group"],
            observed=True,
        )
        .agg(
            MAE=("abs_error", "mean"),
            n=("abs_error", "size"),
        )
        .reset_index()
    )

    grouped_B_ws["seed"] = seed
    B_ws_error_rows.append(grouped_B_ws)

farm_error_by_band = pd.concat(
    farm_error_rows,
    ignore_index=True,
)

B_ws_error_by_band = pd.concat(
    B_ws_error_rows,
    ignore_index=True,
)

farm_standardised = (
    farm_error_by_band
    .groupby(
        ["seed", "model", "wake_group"]
    )["MAE"]
    .mean()
    .reset_index()
)

B_ws_standardised = (
    B_ws_error_by_band
    .groupby(
        ["seed", "wake_group"]
    )["MAE"]
    .mean()
    .reset_index()
)


In [ ]:
# table comparing High and Non-high wake conditions

table_rows = []

for model_name in ["A", "C", "B"]:
    model_data = farm_standardised[
        farm_standardised["model"]
        == model_name
    ]

    pivot = model_data.pivot(
        index="seed",
        columns="wake_group",
        values="MAE",
    )

    difference = (
        pivot["High wake"]
        - pivot["Non-high wake"]
    )

    table_rows.append(
        {
            "metric": (
                f"Model {model_name} farm power"
            ),
            "non_high_mean": (
                pivot["Non-high wake"].mean()
            ),
            "non_high_sd": (
                pivot["Non-high wake"].std(ddof=1)
            ),
            "high_mean": (
                pivot["High wake"].mean()
            ),
            "high_sd": (
                pivot["High wake"].std(ddof=1)
            ),
            "difference_mean": difference.mean(),
            "difference_sd": difference.std(ddof=1),
            "unit": "kW",
        }
    )

B_ws_pivot = B_ws_standardised.pivot(
    index="seed",
    columns="wake_group",
    values="MAE",
)

B_ws_difference = (
    B_ws_pivot["High wake"]
    - B_ws_pivot["Non-high wake"]
)

table_rows.insert(
    2,
    {
        "metric": "Model B turbine wind speed",
        "non_high_mean": (
            B_ws_pivot["Non-high wake"].mean()
        ),
        "non_high_sd": (
            B_ws_pivot["Non-high wake"].std(ddof=1)
        ),
        "high_mean": (
            B_ws_pivot["High wake"].mean()
        ),
        "high_sd": (
            B_ws_pivot["High wake"].std(ddof=1)
        ),
        "difference_mean": (
            B_ws_difference.mean()
        ),
        "difference_sd": (
            B_ws_difference.std(ddof=1)
        ),
        "unit": "m/s",
    },
)

wake_group_comparison = pd.DataFrame(
    table_rows
)

farm_error_by_band.to_csv(
    WAKE_RESULTS_DIR
    / "farm_error_by_wake_and_speed.csv",
    index=False,
)

B_ws_error_by_band.to_csv(
    WAKE_RESULTS_DIR
    / "model_B_ws_error_by_wake_and_speed.csv",
    index=False,
)

wake_group_comparison.to_csv(
    WAKE_RESULTS_DIR
    / "high_vs_non_high_wake_summary.csv",
    index=False,
)

print(
    wake_group_comparison
    .round(4)
    .to_string(index=False)
)


In [ ]:
# key directional values used in the results text

key_directions = [55, 60, 125, 305]

key_direction_values = relative_summary.loc[
    relative_summary["dir_bin"].isin(
        key_directions
    ),
    [
        "dir_bin",
        "Observed",
        "A_mean",
        "B_mean",
        "C_mean",
    ],
].copy()

deepest_directional_reductions = (
    relative_summary
    .nsmallest(5, "Observed")[
        [
            "dir_bin",
            "Observed",
            "A_mean",
            "B_mean",
            "C_mean",
        ]
    ]
    .copy()
)

key_direction_values.to_csv(
    WAKE_RESULTS_DIR
    / "key_directional_relative_power.csv",
    index=False,
)

deepest_directional_reductions.to_csv(
    WAKE_RESULTS_DIR
    / "deepest_directional_reductions.csv",
    index=False,
)

print(key_direction_values.round(3).to_string(index=False))
print()
print(
    deepest_directional_reductions
    .round(3)
    .to_string(index=False)
)
